In [ ]:
import pathlib
import random
import copy
import numpy as np
import torch
import torchvision
from captum.attr import *
import matplotlib.pyplot as plt
from omegaconf import OmegaConf
import yaml
import argparse
from data.data_loader import load_eeg_data, get_sliding_window_data, create_dataloader
import os
import mne
from tqdm import tqdm
from sklearn.preprocessing import MinMaxScaler
import pandas as pd

from sliced_wasserstein import sliced_wasserstein_distance
from c2st import c2st_knn, c2st_nn, c2st_rf

from models.s4net import S4PatchedFinalNet,TrunkNet, HeadNet
import os

In [ ]:
CFG_YAML = """
wandb:
 key: f0c92a0059bf12e2647f0a1c22fdcd12555fa6df
model:
dataset:
 data_directory: /home/marco/Documents/GitHub/tms_eeg_decoding/data
 #file_name: subject_{:03d}_preprocessed_combined_py.fif
 file_name: subject_{:03d}_preprocessed_combined_py.fif
 exclude_timepoints: 100
 subject_index: 1
 test_subject_indices: [1,2,13,24,26,27,29,34,35,41, 42,43,45,46,47,48,52,55,56,57,60,62,67,69,72,73,79,80,86,88,92,102]
 #test_subject_indices: [2]
training:
 training_start_len: 100
 pretrain_epochs: 100
 pretrain_lr: 0.0001
 val_window_len: 1
 epochs_per_window: 10
 num_warmup_epochs: 5
 num_epochs: 800
 slide_step: 1
 num_warmup_epochs_per_window: 0
 lr: 0.005 #maybe change back to 0.0001
 nll_beta: 0.001
 num_warmup_epochs: 0
 batch_size: 50 #better to use 50
 random_seed: 42
 precision: bf16
 kde_lambda: 0.5
 finetune_entire_model: true # Set to true to finetune the entire model, false for transformer only
exp_name: S4_S4EEGNet_ema
"""


def load_config():
    cfg = OmegaConf.create(yaml.safe_load(CFG_YAML))
    cfg.exp_name = f"{cfg.exp_name}_subject_{cfg.dataset.subject_index}"
    return cfg

def parse_args():
    parser = argparse.ArgumentParser()
    parser.add_argument("--update_conf", nargs="*", help="Updates to the configuration in the form of key=value pairs", default=[])
    parser.add_argument("-f", "--fff", help="A dummy argument to handle IPython's default argument", default="1")
    return parser.parse_args()

def update_config(cfg, cli_args):
    for update in cli_args.update_conf:
        key, value = update.split("=")
        try:
            value = eval(value)
        except:
            pass
        OmegaConf.update(cfg, key, value, force_add=True)
    cfg.exp_name = cfg.exp_name + "_" + "_".join(cli_args.update_conf)
    print(OmegaConf.to_yaml(cfg))
    return cfg


def save_config(cfg):
    os.makedirs("conf/sweeps", exist_ok=True)
    os.makedirs("exp/withinsubs", exist_ok=True)
    with open(f"conf/sweeps/withinsubs_{cfg.exp_name}.yaml", "w") as f:
        f.write(OmegaConf.to_yaml(cfg))


In [ ]:
def load_model(cfg, start_index=100, subject_index=2):
    save_path = "/home/marco/Documents/GitHub/tms_eeg_decoding/data/model_checkpoints/finetune"
    
    file_path = os.path.join(save_path, f"subject_{subject_index}", f"model_checkpoint_finetune_subject_index_{subject_index}_start_idx_{start_index}_rep_0_pen.pth") 
    trunk_net = TrunkNet(n_chans=input_shape_st[0], n_times=input_shape_st[1])
    head_net = HeadNet(64, 1)  # Assuming these are the correct dimensions
    model = S4PatchedFinalNet(64, trunk_net, head_net)
    
    weights = torch.load(file_path)
    model.load_state_dict(weights)
    #model.load_state_dict(full_checkpoint['model_state_dict'])
    model.eval()
    model.to(device)
    return model

cfg = load_config()

subject_index = cfg.dataset.test_subject_indices[1]
cfg.dataset.subject_index = subject_index
cfg.exp_name = f"S4_S4EEGNet_ema_100_cal_py_{cfg.dataset.subject_index}"
cli_args = parse_args()
cfg = update_config(cfg, cli_args)
save_config(cfg)
all_epochs, labels_raw, _, _, _, _, _, ch_names = load_eeg_data(cfg)
all_epochs = all_epochs[150:]
labels_raw = labels_raw[150:]
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
input_shape_st = (60, 900)

## perturb all samples

result for the classifiers seem to suggest that the classifieres cant distinguish between the samples. This is as desired as the samples come from the same distribution. Furthermore shuffling the samples potentially had the effect of eliminating any shifts in distribution during the time series.

## check deviation of samples from the original distribution to perturbed samples

## check how deviation from original distribution scales with magnitude of perturbation for alpha frequency band

## perturb all samples all channels for every frequency band and different amplification factors for the given frequency band

In [ ]:
freq_bands = {"delta": (0.5, 4),
            "theta": (4, 8),
              "alpha": (8, 12),
              "beta": (12, 30),
              "gamma": (30, 45)}
phase_peturbations = np.arange(45, 316, 45)

In [ ]:
import matplotlib.pylab as pylab
params = {'legend.fontsize': 'x-large',
          'figure.titlesize': 'x-large',
          'figure.figsize': (15, 5),
         'axes.labelsize': 'x-large',
         'axes.titlesize':'x-large',
         'xtick.labelsize':'x-large',
         'ytick.labelsize':'x-large'}
pylab.rcParams.update(params)

# test how the model prediction changes with the perturbed samples vs the original samples

## get prediction on original samples and perturbed samples

### original predictions

In [ ]:
def load_original_predictions_uncertainties(subject_index=2):
    dir = "/home/marco/Documents/GitHub/tms_eeg_decoding/subject_interpretability"
    file_name = f"subject_{subject_index}_results.pkl"
    file_path = os.path.join(dir, file_name)
    data = np.load(file_path, allow_pickle=True)
    #print(data.keys())
    return data['predictions'], data['uncertainties']


In [ ]:
def load_perturbed_predictions_and_uncertainties(subject_index=2):
    dir = "/home/marco/Documents/GitHub/tms_eeg_decoding/perturb_samples_phase/all_phase_perturb"

    file_name = f"prediction_and_uncertanties_subject_{subject_index}_phase.npy"
    file_path = os.path.join(dir, file_name)
    data = np.load(file_path, allow_pickle=True)
    #print(data.keys())
    return data[0], data[1]

In [ ]:
pred_label_original, uncertainties_original = load_original_predictions_uncertainties(subject_index=2)

In [ ]:
predictions_perturbed, uncertainties_perturbed = load_perturbed_predictions_and_uncertainties(subject_index = 2)

In [ ]:
predictions_perturbed.keys()

### predictions for all perturbed samples

# compare alignment of original predictions with perturbed predictions

In [ ]:
import pandas as pd


# Initialize a DataFrame to store the comparison results
comparison_df = pd.DataFrame(index=phase_peturbations , columns=freq_bands.keys())

# Fill the DataFrame with the mean absolute difference between original and perturbed predictions
for band_name in freq_bands.keys():
    for factor in phase_peturbations :
        original_predictions = pred_label_original
        key = f'{band_name}_{factor}°'
        perturbed_predictions = predictions_perturbed[key]
        mean_abs_diff = np.median(original_predictions - perturbed_predictions)
        comparison_df.loc[factor, band_name] = mean_abs_diff

print(comparison_df)
# Plot the comparison results
fig, ax = plt.subplots(figsize=(12, 8))
comparison_df.plot(kind='bar', ax=ax)
ax.set_title('Median difference between Original and Perturbed Predictions')
ax.set_xlabel('Amplification Factor')
ax.set_ylabel('Median Difference')
plt.xticks(rotation=0)
plt.legend(title='Frequency Bands')
plt.savefig("median_absolute_difference_all_channels_per_frequency_band.png")

# Plot the comparison results
fig, axes = plt.subplots(len(phase_peturbations ), len(freq_bands), figsize=(20, 20))
for i, factor in enumerate(phase_peturbations ):
    for j, band_name in enumerate(freq_bands.keys()):
        key = f'{band_name}_{factor}°'
        perturbed_predictions = predictions_perturbed[key]
        axes[i, j].scatter(pred_label_original, predictions_perturbed[key], alpha=0.5)
        axes[i, j].set_title(f'{band_name} {factor}')
        if i == len(phase_peturbations ) - 1:
            axes[i, j].set_xlabel('Original Predictions')
        if j == 0:
            axes[i, j].set_ylabel('Perturbed Predictions')

plt.tight_layout()
plt.show()

x = np.arange(len(pred_label_original))
# Plot the comparison results
fig, axes = plt.subplots(len(phase_peturbations ), len(freq_bands), figsize=(20, 20))
for i, factor in enumerate(phase_peturbations ):
    for j, band_name in enumerate(freq_bands.keys()):
        key = f'{band_name}_{factor}°'
        perturbed_predictions = predictions_perturbed[key]

            #axes[i, j].scatter(pred_label_original, predictions_perturbed[key], alpha=0.5)
        axes[i, j].scatter(x, pred_label_original, alpha=0.5, c='blue', label=f'original, {np.median(pred_label_original):.2f}', s=4)
        axes[i, j].scatter(x, predictions_perturbed[key], alpha=0.5, c='red', label=f'perturbed, {np.median(predictions_perturbed[key]):.2f}', s=4)

        axes[i, j].set_title(f'{band_name} {factor}')
        if i == len(phase_peturbations) - 1:
            axes[i, j].set_xlabel('trial')
        if j == 0:
            axes[i, j].set_ylabel('predictions')
        axes[i, j].legend()

the power in a frequency band over all channels seem to influence the final prediction of the model for the delta, alpha and the gamma band especially, with the predicted amplitude falling with power in the delta band and growing with power in the gamma, alpha band.

To make sure the predictions are not due to evaluation in extrapolation we checked how OOD the perturbed samples are with SWD and c2st. We also want to see how these are aligned to the uncertainty of the prediction output by the model

also check if the prediction for some samples decreases while the prediction for other samples increases. This may point to interaction-effects of the power in one frequency band with other features

At last,  no change in beta band until the 10x amplification may suggest what we only extrapolate in the 10x case for the beta band
 (or that change in prediction due to extrapolation and change in prediction due to the true different result may balance out?). Furthmore the severity of outliers increases a lot in higher amplification settings. This suggests that some trials are definitely more susceptible to changes in power, again pointing to more complex interaction effects.

# check how well measures of OODness and uncertainity output by the network are aligned

## first check how the OOD metrics introduced in the paper perform

the results seem to suggest that for amplification you can at maxium go a factor of ~3 and at minimum a factor of ~0.5(this needs to be rechecked)
Also as expected, the more we amplify we the power in a frequency band the greater the the SWD becomes. Furthermore the RF classifier seems to be most proficient and presumably closest to the bayes optimal classifier.

## compare the results to the predicted uncertainties for the perturbed samples

In [ ]:

# Initialize a DataFrame to store the comparison results
uncertainty_comparison_df = pd.DataFrame(index=phase_peturbations, columns=freq_bands.keys())

# Fill the DataFrame with the mean absolute difference between original and perturbed uncertainties
for band_name in freq_bands.keys():
    for factor in phase_peturbations:
        original_uncertainties = uncertainties_original
        key = f'{band_name}_{factor}°'
        perturbed_uncertainties = uncertainties_perturbed[key]
        mean_abs_diff = np.median(original_uncertainties - perturbed_uncertainties)
        uncertainty_comparison_df.loc[factor, band_name] = mean_abs_diff

print(uncertainty_comparison_df)

# Plot the comparison results
fig, ax = plt.subplots(figsize=(12, 8))
uncertainty_comparison_df.plot(kind='bar', ax=ax)
ax.set_title('Median Absolute Difference between Original and Perturbed Uncertainties')
ax.set_xlabel('Amplification Factor')
ax.set_ylabel('Median Absolute Difference')

fig.savefig("median_absolute_difference_uncertainties_all_channels_per_frequency_band.png")


# Plot the comparison results
fig, axes = plt.subplots(len(phase_peturbations), len(freq_bands), figsize=(20, 20))
for i, factor in enumerate(phase_peturbations):
    for j, band_name in enumerate(freq_bands.keys()):
        key = f'{band_name}_{factor}°'

        axes[i, j].scatter(uncertainties_original, uncertainties_perturbed[key], alpha=0.5)
        axes[i, j].set_title(f'{band_name} {factor}')
        if i == len(phase_peturbations) - 1:
            axes[i, j].set_xlabel('Original Uncertainties')
        if j == 0:
            axes[i, j].set_ylabel('Perturbed Uncertainties')


x = np.arange(len(uncertainties_original))
# Plot the comparison results
fig, axes = plt.subplots(len(phase_peturbations), len(freq_bands), figsize=(20, 20))
for i, factor in enumerate(phase_peturbations):
    for j, band_name in enumerate(freq_bands.keys()):
        key = f'{band_name}_{factor}°'
        axes[i, j].scatter(x, np.log(uncertainties_original), alpha=0.5, c='blue', label=f'original, {np.median(uncertainties_original):.2f}', s=4)
        axes[i, j].scatter(x, np.log(uncertainties_perturbed[key]), alpha=0.5, c='red', label=f'perturbed, {np.median(uncertainties_perturbed[key]):.2f}', s=4)

        axes[i, j].set_title(f'{band_name} {factor}')
        if i == len(phase_peturbations) - 1:
            axes[i, j].set_xlabel('trial')
        if j == 0:
            axes[i, j].set_ylabel('log uncertainties')
        axes[i, j].legend()
            # axes[i, j].set_ylim(0, 1)   
fig.savefig("log_uncertaintiesper frequency band and amp factor across trials.png")

The first plot seems to suggest that increasing the power in some frequency bands like beta and alpha even decreases the uncertainity (since it is computed via original_uncertainity-perturbed_uncertainty)
, which is not in line with our measures of OODness that suggest that samples are more OOD for higher factors and that should reflect in in the predicted uncertainty.

# perturbed prediction difference

for all subjects

In [ ]:
def calculate_mean_abs_diff(pred_label_original, predictions_perturbed, freq_bands, phase_peturbations, subject_index=2):
    comparison_df = pd.DataFrame(index=phase_peturbations, columns=freq_bands.keys())

    # Fill the DataFrame with the mean absolute difference between original and perturbed predictions
    for band_name in freq_bands.keys():
        for factor in phase_peturbations:
            key = f'{band_name}_{factor}°'
            original_predictions = pred_label_original
            perturbed_predictions = predictions_perturbed[key]
            mean_abs_diff = np.median(original_predictions - perturbed_predictions)
            comparison_df.loc[factor, band_name] = mean_abs_diff


    fig, ax = plt.subplots(figsize=(12, 4))
    fig.suptitle(subject_index)
    comparison_df.plot(kind='bar', ax=ax)
    ax.set_title('Median difference between Original and Perturbed Predictions')
    ax.set_xlabel('Amplification Factor')
    ax.set_ylabel('Median Difference')
    plt.xticks(rotation=0)
    plt.legend(title='Frequency Bands')
    plt.savefig("median_absolute_difference_all_channels_per_frequency_band.png")


# Example usage#
#comparison_df = calculate_mean_abs_diff(pred_label_original, predictions_perturbed, freq_bands, phase_peturbations)
#print(comparison_df)

In [ ]:
def plot_comparison_results(pred_label_original, predictions_perturbed, freq_bands, phase_peturbations, subject_index=2):
    fig, axes = plt.subplots(len(phase_peturbations), len(freq_bands), figsize=(20, 5))
    fig.suptitle(f'Comparison of Original and Perturbed Predictions for Subject {subject_index}')
    for i, factor in enumerate(phase_peturbations):
        for j, band_name in enumerate(freq_bands.keys()):
            key = f'{band_name}_{factor}°'
            axes[i, j].scatter(pred_label_original, predictions_perturbed[key], alpha=0.5)
            axes[i, j].set_title(f'{band_name} {factor}')
            if i == len(phase_peturbations) - 1:
                axes[i, j].set_xlabel('Original Predictions')
            if j == 0:
                axes[i, j].set_ylabel('Perturbed Predictions')

    plt.tight_layout()
    plt.show()

# Example usage
#plot_comparison_results(pred_label_original, predictions_perturbed, freq_bands, phase_peturbations)

In [ ]:
cfg = load_config()
for subject_index in cfg.dataset.test_subject_indices:
    pred_label_original, uncertainties_original = load_original_predictions_uncertainties(subject_index)
    predictions_perturbed, uncertainties_perturbed = load_perturbed_predictions_and_uncertainties(subject_index)
    comparison_df = calculate_mean_abs_diff(pred_label_original, predictions_perturbed, freq_bands, phase_peturbations, subject_index=subject_index)
    #plot_comparison_results(pred_label_original, predictions_perturbed, freq_bands, phase_peturbations, subject_index)

In [ ]:
def calculate_mean_abs_diff_across_subjects1(freq_bands, phase_peturbations):
    # Initialize a DataFrame to store all differences per subject
    all_diffs = {}
    
    for band_name in freq_bands.keys():
        for factor in phase_peturbations:
            all_diffs[(band_name, factor)] = []
    
    colors = {
        'delta': '#E63946',  # Crimson red
        'theta': '#F9A826',  # Orange
        'alpha': '#2A9D8F',  # Teal
        'beta':  '#457B9D',  # Blue
        'gamma': '#6A0DAD',  # Purple
    }
    
    plt.rcParams.update({'font.size': 15}) 
    # Collect data from all subjects
    cfg = load_config()
    for subject_index in cfg.dataset.test_subject_indices:
        try:
            pred_label_original, _ = load_original_predictions_uncertainties(subject_index)
            predictions_perturbed, _ = load_perturbed_predictions_and_uncertainties(subject_index)
            
            for band_name in freq_bands.keys():
                for factor in phase_peturbations:
                    key = f'{band_name}_{factor}°'
                    mean_abs_diff = np.median(pred_label_original - predictions_perturbed[key])
                    all_diffs[(band_name, factor)].append(mean_abs_diff)
        except Exception as e:
            print(f"Error processing subject {subject_index}: {e}")
    
    # Calculate mean and standard deviation across subjects
    mean_diffs = pd.DataFrame(index=phase_peturbations, columns=freq_bands.keys())
    std_diffs = pd.DataFrame(index=phase_peturbations, columns=freq_bands.keys())
    
    for band_name in freq_bands.keys():
        for factor in phase_peturbations:
            if all_diffs[(band_name, factor)]:
                mean_diffs.loc[factor, band_name] = np.mean(all_diffs[(band_name, factor)])
                std_diffs.loc[factor, band_name] = np.std(all_diffs[(band_name, factor)])
            else:
                mean_diffs.loc[factor, band_name] = np.nan
                std_diffs.loc[factor, band_name] = np.nan
    
    # Plot with error bars
    fig, ax = plt.subplots(figsize=(16, 7))
    
    x = np.arange(len(phase_peturbations))
    width = 0.15
    offsets = [-2.0, -1.0, 0.0, 1.0, 2.0]
    
    for i, band_name in enumerate(freq_bands.keys()):
        means = mean_diffs[band_name].values
        errors = std_diffs[band_name].values
        ax.bar(x + offsets[i] * width, means, width, label=band_name, 
               yerr=errors, capsize=5, alpha=0.7, color=colors[band_name])
    
    ax.set_xlabel('phase shift radians', fontsize=18)
    ax.set_ylabel("$I^{\phi,\omega}$", fontsize=18)
    ax.set_title('prediction difference over all subjects', 
                fontsize=20)
    ax.set_xticks(x)
    ax.set_xticklabels(["$\pi/4$", "$\pi/2$", "$3\pi/4$", "$\pi$", "$5\pi/4$", "$3\pi/2$", "$7\pi/4$"], fontsize=16)
    ax.legend(title='Frequency Bands', fontsize=16)
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    
    plt.tight_layout()
    plt.savefig("phase_mean_median_difference_all_subjects.png", dpi=300)
    plt.show()
    
    return mean_diffs, std_diffs

# Call the function
mean_diffs, std_diffs = calculate_mean_abs_diff_across_subjects1(freq_bands, phase_peturbations)
print(mean_diffs)


In [ ]:
def calculate_weighted_mean_abs_diff_across_subjects(freq_bands, phase_peturbations):
    # Initialize a DataFrame to store all differences per subject
    all_diffs = {}
    
    for band_name in freq_bands.keys():
        for factor in phase_peturbations:
            all_diffs[(band_name, factor)] = []
    
    colors = {
        'delta': '#E63946',  # Crimson red
        'theta': '#F9A826',  # Orange
        'alpha': '#2A9D8F',  # Teal
        'beta':  '#457B9D',  # Blue
        'gamma': '#6A0DAD',  # Purple
    }

    plt.rcParams.update({'font.size': 15})  # Increase base font size
    # Collect data from all subjects
    cfg = load_config()
    for subject_index in cfg.dataset.test_subject_indices:
   
        pred_label_original, _ = load_original_predictions_uncertainties(subject_index)
        predictions_perturbed, _ = load_perturbed_predictions_and_uncertainties(subject_index)
            
        for band_name in freq_bands.keys():
            for factor in phase_peturbations:
                key = f'{band_name}_{factor}°'
                mean_abs_diff = np.median(pred_label_original - predictions_perturbed[key])
                all_diffs[(band_name, factor)].append(mean_abs_diff)

        
    
    mean_diffs = pd.DataFrame(index=phase_peturbations, columns=freq_bands.keys())
    std_diffs = pd.DataFrame(index=phase_peturbations, columns=freq_bands.keys())

    weights = {}
    for band_name, (low, high) in freq_bands.items():
        weights[band_name] = high - low

    weighted_mean_diffs = pd.DataFrame(index=phase_peturbations, columns=freq_bands.keys())
    weighted_std_diffs = pd.DataFrame(index=phase_peturbations, columns=freq_bands.keys())
    
    for band_name in freq_bands.keys():
        for factor in phase_peturbations:
            if all_diffs[(band_name, factor)]:
                mean_diffs.loc[factor, band_name] = np.mean(all_diffs[(band_name, factor)])
                std_diffs.loc[factor, band_name] = np.std(all_diffs[(band_name, factor)])
                weighted_mean_diffs.loc[factor, band_name] = np.mean(all_diffs[(band_name, factor)]) / weights[band_name]
                weighted_std_diffs.loc[factor, band_name] = np.std(all_diffs[(band_name, factor)]) / weights[band_name]
            else:
                mean_diffs.loc[factor, band_name] = np.nan
                std_diffs.loc[factor, band_name] = np.nan
                weighted_mean_diffs.loc[factor, band_name] = np.nan
                weighted_std_diffs.loc[factor, band_name] = np.nan
    

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 8), sharex=True)
    
    x = np.arange(len(phase_peturbations))
    width = 0.15
    offsets = [-2.0, -1.0, 0.0, 1.0, 2.0] 
    
    for i, band_name in enumerate(freq_bands.keys()):
        means = mean_diffs[band_name].values
        errors = std_diffs[band_name].values
        ax1.bar(x + offsets[i] * width, means, width, label=f"{band_name}", 
               yerr=errors, capsize=5, color=colors[band_name], alpha=0.8)
    
    ax1.set_xlabel('Phase shift in radians', fontsize=20)
    ax1.set_ylabel("$I^{\Delta \phi,\omega}$", fontsize="18")
    ax1.set_title('Unweighted prediction difference over all subjects', 
                fontsize=16)
    ax1.set_xticks(x)

    ax1.set_xticklabels(["$\pi/4$", "$\pi/2$", "$3\pi/4$", "$\pi$", "$5\pi/4$", "$3\pi/2$", "$7\pi/4$"], fontsize=16)
    ax1.tick_params(axis='both', which='major', labelsize=19)
    ax1.legend(title='Frequency Bands', prop={'size': 14}, title_fontsize=13, loc="upper right")
    ax1.grid(axis='y', linestyle='--', alpha=0.7)
    

    for i, band_name in enumerate(freq_bands.keys()):
        weighted_means = weighted_mean_diffs[band_name].values
        weighted_errors = weighted_std_diffs[band_name].values
        ax2.bar(x + offsets[i] * width, weighted_means, width, 
               label=f"{band_name}", color=colors[band_name],
               yerr=weighted_errors, capsize=5, alpha=0.8)
    
    ax2.set_xlabel('Phase shift in radians', fontsize=18)
    #ax2.set_ylabel('Weighted prediction difference', fontsize=18)
    ax2.set_title('Frequency-range weighted prediction difference over all subjects', 
                fontsize=16)
    ax2.set_xticks(x)
    ax2.set_xticklabels(phase_peturbations)
    ax2.set_xticklabels(["$\pi/4$", "$\pi/2$", "$3\pi/4$", "$\pi$", "$5\pi/4$", "$3\pi/2$", "$7\pi/4$"], fontsize=16)
    ax2.tick_params(axis='both', which='major', labelsize=19)

    ax2.grid(axis='y', linestyle='--', alpha=0.7)
    
    plt.tight_layout()
    plt.savefig("phase_weighted_vs_unweighted_differences_all_subjects.png", dpi=300)
    plt.show()
    

    weight_df = pd.DataFrame({
        'Band': freq_bands.keys(),
        'Range (Hz)': [f"{low}-{high}" for low, high in freq_bands.values()],
        'Width (Hz)': [high - low for low, high in freq_bands.values()],
        'Weight': [weights[band] for band in freq_bands.keys()]
    })
    
    print("\nFrequency band weights:")
    print(weight_df)
    
    return mean_diffs, weighted_mean_diffs, std_diffs, weights

# Call the function
mean_diffs, weighted_diffs, std_diffs, weights = calculate_weighted_mean_abs_diff_across_subjects(freq_bands, phase_peturbations)
print("\nUnweighted mean differences:")
print(mean_diffs)
print("\nWeighted mean differences:")
print(weighted_diffs)


gamma band seems consistently to be the most important. Not that direction of power change is variable and results are somewhat questionable looking at OODness. in the other hand changes seem to behave linearly within subject. direction of changes varies between subjects but (except for one subject) are conistently in one direction

# uncertainty difference

In [ ]:
def plot_uncertainty_comparison(uncertainties_original, uncertainties_perturbed, freq_bands, phase_peturbations, subject_index=2):
    uncertainty_comparison_df = pd.DataFrame(index=phase_peturbations, columns=freq_bands.keys())

    # Fill the DataFrame with the mean absolute difference between original and perturbed uncertainties
    for band_name in freq_bands.keys():
        for factor in phase_peturbations:
            key = f'{band_name}_{factor}°'
            original_uncertainties = uncertainties_original
            perturbed_uncertainties = uncertainties_perturbed[key]
            mean_abs_diff = np.median(original_uncertainties - perturbed_uncertainties)
            uncertainty_comparison_df.loc[factor, band_name] = mean_abs_diff

    #print(uncertainty_comparison_df)

    # Plot the comparison results
    fig, ax = plt.subplots(figsize=(12, 4))
    fig.suptitle(f'Subject {subject_index}')
    uncertainty_comparison_df.plot(kind='bar', ax=ax)
    ax.set_title('Median Absolute Difference between Original and Perturbed Uncertainties')
    ax.set_xlabel('Amplification Factor')
    ax.set_ylabel('Median Absolute Difference')


# Example usage
#plot_uncertainty_comparison(uncertainties_original, uncertainties_perturbed, freq_bands, phase_peturbations)

In [ ]:
def plot_log_uncertainties(x, uncertainties_original, uncertainties_perturbed, freq_bands, phase_peturbations, subject_index=2):
    fig, axes = plt.subplots(len(phase_peturbations), len(freq_bands), figsize=(20, 10))
    fig.suptitle(f'Subject {subject_index}')
    for i, factor in enumerate(phase_peturbations):
        for j, band_name in enumerate(freq_bands.keys()):
            key = f'{band_name}_{factor}°'
            axes[i, j].scatter(x, np.log(uncertainties_original), alpha=0.5, c='blue', label=f'original, {np.median(uncertainties_original):.2f}', s=4)
            axes[i, j].scatter(x, np.log(uncertainties_perturbed[key]), alpha=0.5, c='red', label=f'perturbed, {np.median(uncertainties_perturbed[key]):.2f}', s=4)

            axes[i, j].set_title(f'{band_name} {factor}')
            if i == len(phase_peturbations) - 1:
                axes[i, j].set_xlabel('trial')
            if j == 0:
                axes[i, j].set_ylabel('log uncertainties')
            axes[i, j].legend()


# Example usage
#plot_log_uncertainties(x, uncertainties_original, uncertainties_perturbed, freq_bands, phase_peturbations)

In [ ]:
def calculate_mean_uncertainty_diff_across_subjects(freq_bands, phase_peturbations):
    # Initialize a DataFrame to store all differences per subject
    all_uncertainty_diffs = {}
    
    for band_name in freq_bands.keys():
        for factor in phase_peturbations:
            all_uncertainty_diffs[(band_name, factor)] = []
    
    # Collect data from all subjects
    cfg = load_config()
    for subject_index in cfg.dataset.test_subject_indices:
        try:
            _, uncertainties_original = load_original_predictions_uncertainties(subject_index)
            _, uncertainties_perturbed = load_perturbed_predictions_and_uncertainties(subject_index)
            
            for band_name in freq_bands.keys():
                for factor in phase_peturbations:
                    key = f'{band_name}_{factor}°'
                    # Calculate median difference in uncertainties
                    median_diff = np.median(uncertainties_original - uncertainties_perturbed[key])
                    all_uncertainty_diffs[(band_name, factor)].append(median_diff)
        except Exception as e:
            print(f"Error processing subject {subject_index}: {e}")
    
    # Calculate mean and standard deviation across subjects
    mean_uncertainty_diffs = pd.DataFrame(index=phase_peturbations, columns=freq_bands.keys())
    std_uncertainty_diffs = pd.DataFrame(index=phase_peturbations, columns=freq_bands.keys())
    
    for band_name in freq_bands.keys():
        for factor in phase_peturbations:
            if all_uncertainty_diffs[(band_name, factor)]:
                mean_uncertainty_diffs.loc[factor, band_name] = np.mean(all_uncertainty_diffs[(band_name, factor)])
                std_uncertainty_diffs.loc[factor, band_name] = np.std(all_uncertainty_diffs[(band_name, factor)])
            else:
                mean_uncertainty_diffs.loc[factor, band_name] = np.nan
                std_uncertainty_diffs.loc[factor, band_name] = np.nan
    
    colors = {
        'delta': '#E63946',  # Crimson red
        'theta': '#F9A826',  # Orange
        'alpha': '#2A9D8F',  # Teal
        'beta':  '#457B9D',  # Blue
        'gamma': '#6A0DAD',  # Purple
    }
    

    # Plot with error bars
    fig, ax = plt.subplots(figsize=(16, 7))
    
    x = np.arange(len(phase_peturbations))
    width = 0.15
    offsets = [-2.0, -1.0, 0.0, 1.0, 2.0] 
    
    for i, band_name in enumerate(freq_bands.keys()):
        means = mean_uncertainty_diffs[band_name].values
        errors = std_uncertainty_diffs[band_name].values
        ax.bar(x + offsets[i] * width, means, width, label=band_name, 
               yerr=errors, capsize=5, alpha=0.7, color=colors[band_name])
    
    ax.set_xlabel('phase shift in radians', fontsize=18)
    ax.set_ylabel('Uncertainty difference', fontsize=18)
    ax.set_title('Mean uncertainty difference across all subjects', 
                fontsize=18)
    ax.set_xticks(x)
    ax.set_xticklabels(["$\pi/4$", "$\pi/2$", "$3\pi/4$", "$\pi$", "$5\pi/4$", "$3\pi/2$", "$7\pi/4$"], fontsize=20)
    ax.legend(title='Frequency Bands', fontsize=16, ncol=len(freq_bands), loc='upper center', bbox_to_anchor=(0.5, -0.15))
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    
    plt.tight_layout()
    plt.savefig("phase_mean_uncertainty_difference_all_subjects.png", dpi=300,bbox_inches='tight')
    plt.show()
    
    # Also create a separate plot with log scale for better visibility of small differences
    fig, ax = plt.subplots(figsize=(16, 8))
    
    for i, band_name in enumerate(freq_bands.keys()):
        means = np.abs(mean_uncertainty_diffs[band_name].values)
        errors = std_uncertainty_diffs[band_name].values
        ax.bar(x + offsets[i] * width, means, width, label=band_name, 
               yerr=errors, capsize=5, alpha=0.7)
    
    ax.set_xlabel('phase shift radians', fontsize=18)
    ax.set_ylabel('Absolute uncertainty difference', fontsize=18)
    ax.set_title('Mean absolute uncertainty difference across all subjects', 
                fontsize=20)
    ax.set_xticks(x)
    ax.set_xticklabels(phase_peturbations)
    ax.legend(title='Frequency Bands', fontsize=15)
    #ax.set_yscale('log')
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    
    plt.tight_layout()
    plt.savefig("phase_mean_abs_uncertainty_difference_all_subjects_log.png", dpi=300, bbox_inches='tight')
    plt.show()
    
    return mean_uncertainty_diffs, std_uncertainty_diffs

# Call the function
mean_uncertainty_diffs, std_uncertainty_diffs = calculate_mean_uncertainty_diff_across_subjects(freq_bands, phase_peturbations)
print("\nMean uncertainty differences:")
print(mean_uncertainty_diffs)

In [ ]:
cfg = load_config()
for subject_index in cfg.dataset.test_subject_indices:
    pred_label_original, uncertainties_original = load_original_predictions_uncertainties(subject_index)
    predictions_perturbed, uncertainties_perturbed = load_perturbed_predictions_and_uncertainties(subject_index)
    plot_uncertainty_comparison(uncertainties_original, uncertainties_perturbed, freq_bands, phase_peturbations, subject_index)
    x = np.arange(len(uncertainties_original))
    plot_log_uncertainties(x, uncertainties_original, uncertainties_perturbed, freq_bands, phase_peturbations, subject_index)